# Stock Market Scanners

## How to use
| Step | Cell | When to run |
|------|------|-------------|
| 1 | **Install** | Once per session |
| 2 | **SGX Download** | Once — saves data to CSV |
| 3 | **SGX Scanner** | Anytime — loads from memory or CSV |
| 4 | **S&P 500 Download** | Once — saves data to CSV |
| 5 | **S&P 500 Scanner** | Anytime — loads from memory or CSV |
| 6 | **S&P 500 RSI Scanner** | Anytime — loads from memory or CSV |

> **Tip:** After running a Download cell, all scanner cells below it  
> will reuse the data already in memory — no re-download needed.

## Cell 1 — Install
Run once per Colab session.

In [ ]:
!pip install yfinance requests lxml -q

---
## Cell 2 — SGX Download
Downloads all SGX stocks + STI benchmark. Saves to `/content/`. Run once, then use the scanner without re-downloading.

In [ ]:
# ============================================================
# SGX DOWNLOAD — saves sgx_historical.csv + sti_close.csv
# Run once per session. Scanner cells load from these files.
# ============================================================
import csv, time, warnings
import requests, pandas as pd, yfinance as yf
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")

SGX_HIST_CSV = "/content/sgx_historical.csv"
STI_CSV      = "/content/sti_close.csv"
BATCH_SIZE   = 50
BENCHMARK    = "^STI"

print("=" * 55)
print("  SGX DATA DOWNLOAD")
print("=" * 55)

# ── Fetch SGX stock list ─────────────────────────────────────
print("\n[1/3] Fetching SGX stock list...")
resp = requests.get("https://api.sgx.com/securities/v1.1/",
                    headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
resp.raise_for_status()
stocks = [s for s in resp.json()["data"]["prices"] if s.get("type") == "stocks"]
sgx_ticker_info = {
    f"{s['nc']}.SI": {
        "symbol": s["nc"], "name": s.get("n", ""), "market": s.get("m", ""),
        "sgx_pe": s.get("pe"), "sgx_pb": s.get("pb"),
        "sgx_yld": s.get("yld") or s.get("dy"),
        "sgx_eps": s.get("eps") or s.get("es"),
        "sgx_mc":  s.get("mc")  or s.get("mktcap"),
    }
    for s in stocks if s.get("nc")
}
print(f"    Found {len(sgx_ticker_info)} stocks")

# ── Download price history ───────────────────────────────────
end_date   = datetime.today().strftime("%Y-%m-%d")
start_date = (datetime.today() - timedelta(days=365)).strftime("%Y-%m-%d")
print(f"\n[2/3] Downloading data ({start_date} → {end_date})...")

# STI benchmark
print(f"    Fetching benchmark {BENCHMARK}...")
sti_raw   = yf.download(BENCHMARK, start=start_date, end=end_date, auto_adjust=True, progress=False)
sti_close = sti_raw["Close"].squeeze()
sti_close.to_csv(STI_CSV)

all_tickers = list(sgx_ticker_info.keys())
batches     = [all_tickers[i:i+BATCH_SIZE] for i in range(0, len(all_tickers), BATCH_SIZE)]
sgx_stock_data = {}

with open(SGX_HIST_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["symbol", "name", "market", "date", "open", "high", "low", "close", "volume"])
    for i, batch in enumerate(batches):
        print(f"    Batch {i+1}/{len(batches)}...", end=" ", flush=True)
        try:
            raw = yf.download(batch, start=start_date, end=end_date,
                              auto_adjust=True, progress=False, threads=True)
            results = {}
            if len(batch) == 1:
                if not raw.empty:
                    results[batch[0]] = raw.rename(columns=str.lower)
            else:
                for t in batch:
                    try:
                        df = raw.xs(t, level=1, axis=1).dropna(how="all")
                        if not df.empty:
                            results[t] = df.rename(columns=str.lower)
                    except KeyError:
                        pass
            for yf_t, df in results.items():
                info = sgx_ticker_info[yf_t]
                sgx_stock_data[yf_t] = df
                for date, row in df.iterrows():
                    writer.writerow([
                        info["symbol"], info["name"], info["market"],
                        date.strftime("%Y-%m-%d"),
                        round(float(row.get("open",  0) or 0), 4),
                        round(float(row.get("high",  0) or 0), 4),
                        round(float(row.get("low",   0) or 0), 4),
                        round(float(row.get("close", 0) or 0), 4),
                        int(row.get("volume", 0) or 0),
                    ])
            print(f"ok ({len(results)}/{len(batch)})")
        except Exception as e:
            print(f"ERROR: {e}")
        time.sleep(0.3)

print(f"\n[3/3] Done!")
print(f"    Stocks downloaded : {len(sgx_stock_data)}")
print(f"    Saved to          : {SGX_HIST_CSV}")
print(f"    STI benchmark     : {STI_CSV}")
print("    ✅ Scanner cells can now use this data without re-downloading.")

## Cell 3 — SGX Master Buy Signal Scanner
Ports three signals from **master_buy.pine** onto the SGX universe:
- **Signal A** — Descending Peak Trendline Break
- **Signal C** — MACD Histogram Green Peak
- **Signal D** — Double Peak Pullback to SMA20

Export includes a `signals` column showing which of A / C / D fired.

**Loads from memory -> CSV -> error** (run Cell 2 first).

In [ ]:
# ============================================================
# CELL 3 — SGX MASTER BUY SIGNAL SCANNER
#
# Ports three signals from master_buy.pine:
#   Signal A — Descending Peak Trendline Break
#   Signal C — MACD Histogram Reverse Above Zero (Green Peak)
#   Signal D — Double Peak Pullback to SMA20
#
# Filter : volume > 100,000  AND  price > S$0.20
# Loads  : memory -> sgx_historical.csv -> error (run Cell 2 first)
# Outputs: /content/sgx_buy_signals.csv
#          /content/sgx_buy_signals_watchlist.txt
# ============================================================
import csv as _csv, os, warnings
import numpy as np, pandas as pd

warnings.filterwarnings("ignore")

SGX_HIST_CSV = "/content/sgx_historical.csv"
OUTPUT_CSV   = "/content/sgx_buy_signals.csv"
OUTPUT_WL    = "/content/sgx_buy_signals_watchlist.txt"
MIN_VOLUME, MIN_PRICE = 100_000, 0.20

# Signal A parameters
A_LB_BARS  = 120
A_BODY_PCT = 0.02
A_PULL_PCT = 0.03
A_WING     = 5

# Signal D parameters
D_LOOKBACK  = 12
D_PK2_PCT   = 0.10
D_LOW_BAND  = 0.02
D_MAX_BELOW = 3
D_MOD_PCT   = 0.03

# ============================================================
# HELPERS
# ============================================================
def _ema(series, span):
    return series.ewm(span=span, adjust=False, min_periods=span).mean()

def _sma(series, length):
    return series.rolling(length, min_periods=length).mean()

def _macd(close, fast=12, slow=26, sig=9):
    line   = _ema(close, fast) - _ema(close, slow)
    signal = _ema(line, sig)
    return line, signal, line - signal


# ============================================================
# SIGNAL A -- Descending Peak Trendline Break
# ============================================================
def check_signal_a(df):
    high  = df["high"].values
    low   = df["low"].values
    close = df["close"].values
    open_ = df["open"].values
    n     = len(df)

    if n < A_LB_BARS + A_WING + 2:
        return False

    _, _, hist_s = _macd(df["close"])
    if pd.isna(hist_s.iloc[-1]) or float(hist_s.iloc[-1]) <= 0:
        return False

    c, o, l = float(close[-1]), float(open_[-1]), float(low[-1])
    if not (c > o and (c - o) / o >= A_BODY_PCT):
        return False

    pivots = []
    for i in range(A_WING + 1, min(A_LB_BARS - A_WING, n - A_WING)):
        pos = n - 1 - i
        if pos < A_WING or pos + A_WING >= n:
            continue
        ref = high[pos]
        ok = all(
            (pos + j >= n or high[pos + j] < ref) and
            (pos - j < 0  or high[pos - j] < ref)
            for j in range(1, A_WING + 1)
        )
        if not ok:
            continue
        threshold = ref * (1 - A_PULL_PCT)
        if any(n - 1 - j >= 0 and low[n - 1 - j] <= threshold for j in range(1, i)):
            pivots.append((i, ref))

    if len(pivots) < 2:
        return False

    pB_bar, pB_val = pivots[0]
    pA_bar = pA_val = None
    for pbar, pval in pivots[1:]:
        if pval > pB_val:
            pA_bar, pA_val = pbar, pval
            break

    if pA_bar is None:
        return False

    slope     = (pB_val - pA_val) / float(pA_bar - pB_bar)
    trend_now = pB_val + slope * float(pB_bar)

    return (l < trend_now and c > trend_now and
            pB_val * 0.9 <= c <= pB_val * 1.5)


# ============================================================
# SIGNAL C -- MACD Histogram Reverse Above Zero (Green Peak)
# ============================================================
def check_signal_c(df):
    close = df["close"]
    high  = df["high"].values
    low   = df["low"].values
    open_ = df["open"].values
    n     = len(df)

    if n < 30:
        return False

    ml, _, hist_s = _macd(close)
    sma10 = _sma(close, 10)
    sma20 = _sma(close, 20)
    sma50 = _sma(close, 50)

    needed = [hist_s.iloc[-1], hist_s.iloc[-2], hist_s.iloc[-3],
              hist_s.iloc[-4], hist_s.iloc[-5], hist_s.iloc[-6],
              ml.iloc[-1], sma10.iloc[-1], sma20.iloc[-1], sma50.iloc[-1]]
    if any(pd.isna(v) for v in needed):
        return False

    h0 = float(hist_s.iloc[-1])
    h1 = float(hist_s.iloc[-2])
    h2 = float(hist_s.iloc[-3])
    h3 = float(hist_s.iloc[-4])
    h4 = float(hist_s.iloc[-5])
    h5 = float(hist_s.iloc[-6])
    c_  = float(close.iloc[-1])
    h_  = float(high[-1])
    l_  = float(low[-1])
    s10 = float(sma10.iloc[-1])
    s20 = float(sma20.iloc[-1])
    s50 = float(sma50.iloc[-1])

    return (
        h0 > 0 and h1 > 0 and h2 > 0 and
        abs(h1) < abs(h0) and abs(h1) < abs(h2) and
        float(ml.iloc[-1]) > 0 and
        c_ > float(high[-2]) and c_ > float(high[-3]) and
        any(float(close.iloc[-k]) < float(open_[-k]) for k in range(1, 5)) and
        not (h1 < h2 and h2 < h3 and h3 < h4 and h4 < h5) and
        s10 > s20 and s20 >= s50 and
        c_ > s20 and
        (h_ + l_) / 2 <= s10 * 1.1
    )


# ============================================================
# SIGNAL D -- Double Peak Pullback to SMA20
# ============================================================
def check_signal_d(df):
    close  = df["close"].values
    open_  = df["open"].values
    high   = df["high"].values
    low    = df["low"].values
    volume = df["volume"].values
    n      = len(df)

    if n < max(50, D_LOOKBACK + 5):
        return False

    sma10_s  = _sma(df["close"], 10)
    sma20_s  = _sma(df["close"], 20)
    volSma20 = _sma(df["volume"], 20)
    ml, _, _ = _macd(df["close"])

    if any(pd.isna(v) for v in [sma10_s.iloc[-1], sma20_s.iloc[-1], ml.iloc[-1]]):
        return False

    s10 = float(sma10_s.iloc[-1])
    s20 = float(sma20_s.iloc[-1])
    c   = float(close[-1])
    o   = float(open_[-1])
    h   = float(high[-1])

    if not (c > o and s10 > s20 and s10 <= s20 * (1 + D_MOD_PCT)):
        return False

    peak_highs = [high[n - 1 - i] for i in range(1, D_LOOKBACK) if n - 1 - i >= 0]
    if not peak_highs:
        return False
    d_pk2Val = max(peak_highs)

    d_pk2Bar = 0
    for i in range(1, D_LOOKBACK):
        if n - 1 - i >= 0 and high[n - 1 - i] >= d_pk2Val * 0.9999:
            d_pk2Bar = i
            break

    if d_pk2Bar < 3:
        return False

    if not (h >= d_pk2Val and h <= d_pk2Val * (1 + D_PK2_PCT)):
        return False

    d_minLow      = c
    d_minLowSMA20 = s20
    d_belowCnt    = 0
    d_hasLowVol   = False

    for i in range(1, d_pk2Bar):
        idx = n - 1 - i
        if idx < 0:
            break
        if low[idx] < d_minLow:
            d_minLow      = low[idx]
            d_minLowSMA20 = float(sma20_s.iloc[idx]) if not pd.isna(sma20_s.iloc[idx]) else s20
        if not pd.isna(sma20_s.iloc[idx]) and close[idx] < float(sma20_s.iloc[idx]):
            d_belowCnt += 1
        if not pd.isna(volSma20.iloc[idx]) and volume[idx] < float(volSma20.iloc[idx]):
            d_hasLowVol = True

    d_lowestOk = (d_minLow >= d_minLowSMA20 * (1 - D_LOW_BAND) and
                  d_minLow <= d_minLowSMA20 * (1 + D_LOW_BAND))

    if not (d_lowestOk and d_belowCnt <= D_MAX_BELOW and d_hasLowVol):
        return False

    ml_peak_idx = n - 1 - d_pk2Bar
    ml_peak = float(ml.iloc[ml_peak_idx]) if 0 <= ml_peak_idx < n and not pd.isna(ml.iloc[ml_peak_idx]) else float(ml.iloc[-1])

    return float(ml.iloc[-1]) > ml_peak


# ============================================================
# DATA LOADING  (memory -> sgx_historical.csv -> error)
# ============================================================
print("=" * 65)
print("  SGX MASTER BUY SIGNAL SCANNER  (Signals A / C / D)")
print("=" * 65)

try:
    _ = sgx_stock_data, sgx_ticker_info
    print("\n  Using in-memory SGX data.")
except NameError:
    if os.path.exists(SGX_HIST_CSV):
        print("\n  Loading from CSV...")
        df_all = pd.read_csv(SGX_HIST_CSV, parse_dates=["date"])
        sgx_ticker_info = {}
        sgx_stock_data  = {}
        for sym, grp in df_all.groupby("symbol"):
            yf_t = f"{sym}.SI"
            grp  = grp.set_index("date").sort_index()
            sgx_stock_data[yf_t]  = grp[["open", "high", "low", "close", "volume"]]
            sgx_ticker_info[yf_t] = {
                "symbol": sym,
                "name":   grp["name"].iloc[0],
                "market": grp["market"].iloc[0],
            }
        print(f"  Loaded {len(sgx_stock_data)} stocks.")
    else:
        raise RuntimeError("SGX data not found — run Cell 2 first.")

# ============================================================
# SCAN
# ============================================================
print(f"\n  Scanning {len(sgx_stock_data)} stocks...")

confirmed = []

for yf_t, df in sgx_stock_data.items():
    if len(df) < 60:
        continue
    info       = sgx_ticker_info.get(yf_t, {})
    last_close = float(df["close"].iloc[-1])
    last_vol   = float(df["volume"].iloc[-1])

    if last_vol < MIN_VOLUME or last_close < MIN_PRICE:
        continue

    sig_a = check_signal_a(df)
    sig_c = check_signal_c(df)
    sig_d = check_signal_d(df)

    if not (sig_a or sig_c or sig_d):
        continue

    signals = "+".join(s for s, f in [("A", sig_a), ("C", sig_c), ("D", sig_d)] if f)

    confirmed.append({
        "symbol":     info.get("symbol", yf_t),
        "name":       info.get("name", ""),
        "market":     info.get("market", ""),
        "signals":    signals,
        "signal_A":   bool(sig_a),
        "signal_C":   bool(sig_c),
        "signal_D":   bool(sig_d),
        "last_close": round(last_close, 4),
        "last_vol":   int(last_vol),
        "last_date":  df.index[-1].strftime("%Y-%m-%d"),
    })

confirmed.sort(key=lambda r: (-(int(r["signal_A"]) + int(r["signal_C"]) + int(r["signal_D"])), r["symbol"]))

# ============================================================
# OUTPUT
# ============================================================
print("\n" + "=" * 90)
print(f"  SGX MASTER BUY SIGNALS -- {len(confirmed)} stocks matched")
print("  A=Descending Trendline Break  C=MACD Green Peak  D=Double Peak SMA20")
print("=" * 90)
if confirmed:
    print(f"  {'Sig':<7} {'Symbol':<8} {'Name':<30} {'Market':<10} {'Close':>8} {'Volume':>12}")
    print(f"  {'-'*7} {'-'*8} {'-'*30} {'-'*10} {'-'*8} {'-'*12}")
    for r in confirmed:
        print(f"  {r['signals']:<7} {r['symbol']:<8} {r['name'][:30]:<30} "
              f"{r['market'][:10]:<10} {r['last_close']:>8.4f} {r['last_vol']:>12,}")
else:
    print("  No stocks matched today.")
print("=" * 90)

fields = ["symbol", "name", "market", "signals", "signal_A", "signal_C", "signal_D",
          "last_close", "last_vol", "last_date"]
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = _csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(confirmed)

with open(OUTPUT_WL, "w", encoding="utf-8") as f:
    for r in confirmed:
        f.write(f"{r['symbol']}\n")

print(f"\n  Results -> {OUTPUT_CSV}  ({len(confirmed)} rows)")
print(f"  TV list -> {OUTPUT_WL}  ({len(confirmed)} tickers)")

sgx_matched = confirmed


## Cell 3-FA — SGX FA Check
Fundamental analysis for stocks matched by the SGX Master Buy Scanner (Cell 3).
Fetches 10 FA indicators via yfinance. Exports to Excel with pink highlighting for bad metrics.

In [16]:
# ============================================================
# SGX FA CHECK  (Cell 3b-FA)
# Reads confirmed stocks from Cell 3 (SGX Scanner)
# Output: /content/sgx_fa_check.xlsx
# ============================================================
import pandas as pd, yfinance as yf, time, os, warnings
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
warnings.filterwarnings("ignore")

FA_OUTPUT = "/content/sgx_fa_check.xlsx"

THRESHOLDS = {
    "EPS Growth (%)": (">=", 0),
    "Rev Growth (%)": (">",  10),
    "ROE (%)":        (">",  9),
    "ROIC (%)":       (">=", 15),
    "P/FCF":          ("<",  35),
    "TTM PE":         None,
    "Forward PE":     None,
    "Net Margin (%)": (">",  5),
    "Debt/Equity":    ("<",  1.0),
    "Rule of 40 (%)": (">",  40),
}
PINK = PatternFill(start_color="FFB6C1", end_color="FFB6C1", fill_type="solid")

def _is_bad(col, val):
    if val is None or THRESHOLDS.get(col) is None:
        return False
    op, thr = THRESHOLDS[col]
    try:
        v = float(val)
        return v < thr if op in (">=", ">") else v >= thr
    except: return False

def fetch_fa(yf_ticker):
    res = {c: None for c in THRESHOLDS}
    try:
        t    = yf.Ticker(yf_ticker)
        info = t.info
        eg   = info.get("earningsGrowth");  rg = info.get("revenueGrowth")
        roe  = info.get("returnOnEquity");  mg = info.get("profitMargins")
        de   = info.get("debtToEquity")
        ttm  = info.get("trailingPE");      fwd = info.get("forwardPE")
        mc   = info.get("marketCap");       fcf = info.get("freeCashflow")
        res["EPS Growth (%)"] = round(eg * 100, 1)  if eg  is not None else None
        res["Rev Growth (%)"] = round(rg * 100, 1)  if rg  is not None else None
        res["ROE (%)"]        = round(roe* 100, 1)  if roe is not None else None
        res["Net Margin (%)"] = round(mg * 100, 1)  if mg  is not None else None
        res["Debt/Equity"]    = round(de / 100, 2)  if de  is not None else None
        res["TTM PE"]         = round(ttm, 1)        if ttm is not None else None
        res["Forward PE"]     = round(fwd, 1)        if fwd is not None else None
        res["P/FCF"]          = round(mc / fcf, 1)  if (mc and fcf and fcf > 0) else None
        try:
            bs = t.balance_sheet; fi = t.financials
            if not bs.empty and not fi.empty:
                ni_r  = [r for r in fi.index if "Net Income" in str(r)]
                eq_r  = [r for r in bs.index if any(k in str(r) for k in ("Stockholder","Common Stock Equity","Total Equity"))]
                ltd_r = [r for r in bs.index if "Long Term Debt" in str(r)]
                if ni_r and eq_r:
                    ni  = float(fi.loc[ni_r[0]].iloc[0])
                    eq  = float(bs.loc[eq_r[0]].iloc[0])
                    ltd = float(bs.loc[ltd_r[0]].iloc[0]) if ltd_r else 0
                    ic  = eq + ltd
                    if ic > 0: res["ROIC (%)"] = round(ni / ic * 100, 1)
        except: pass
        if res["Rev Growth (%)"] is not None and res["Net Margin (%)"] is not None:
            res["Rule of 40 (%)"] = round(res["Rev Growth (%)"] + res["Net Margin (%)"], 1)
    except: pass
    return res

# ── Load scan list ────────────────────────────────────────────
try:
    scan_list = sgx_matched
    print(f"✅ Using {len(scan_list)} stocks from SGX Scanner (Cell 3).")
except NameError:
    csv_path = "/content/sgx_buy_signals.csv"
    if os.path.exists(csv_path):
        _df = pd.read_csv(csv_path)
        scan_list = _df.to_dict("records")
        print(f"📂 Loaded {len(scan_list)} confirmed stocks from {csv_path}.")
    else:
        print("❌ No data found — run Cell 3 first."); scan_list = []

# ── FA Fetch ──────────────────────────────────────────────────
if not scan_list:
    print("Nothing to analyse.")
else:
    print(f"\n  Fetching FA data for {len(scan_list)} stocks (SGX) ...\n")
    rows = []
    for i, r in enumerate(scan_list):
        sym = r["symbol"]; yf_t = f"{sym}.SI"
        print(f"  [{i+1:>2}/{len(scan_list)}] {sym:<8}", end=" ", flush=True)
        fa  = fetch_fa(yf_t)
        row = {"Symbol": sym, "Name": r.get("name","")[:24],
               "Close": r.get("last_close"), **fa}
        rows.append(row); time.sleep(0.4)
        print(f"ROE={fa.get('ROE (%)')}%  Margin={fa.get('Net Margin (%)')}%  D/E={fa.get('Debt/Equity')}")

    def _passes_filter(row):
        de    = row.get("Debt/Equity")
        nm    = row.get("Net Margin (%)")
        roic  = row.get("ROIC (%)")
        rev_g = row.get("Rev Growth (%)")
        eps_g = row.get("EPS Growth (%)")
        if any(v is None for v in [de, nm, roic, rev_g, eps_g]):
            return False
        return de < 1 and nm >= 10 and roic > 5 and rev_g > 0 and eps_g > 0

    before = len(rows)
    rows   = [r for r in rows if _passes_filter(r)]
    print(f"\n  FA Filter: {before} → {len(rows)} stocks pass (D/E<1, Margin≥10%, ROIC>5%, RevGrow>0%, EPS>0%)")

    if not rows:
        print("  No stocks passed the FA filter.")
    else:
        df = pd.DataFrame(rows)
        df.to_excel(FA_OUTPUT, index=False, engine="openpyxl")
        wb = load_workbook(FA_OUTPUT); ws = wb.active; ws.title = "SGX FA"
        for c in range(1, ws.max_column + 1):
            cell = ws.cell(row=1, column=c)
            cell.fill = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
            cell.font = Font(bold=True, color="FFFFFF", size=10)
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        ws.row_dimensions[1].height = 32
        col_names = list(df.columns)
        for ri in range(2, ws.max_row + 1):
            for ci, cname in enumerate(col_names, 1):
                cell = ws.cell(row=ri, column=ci)
                if _is_bad(cname, cell.value): cell.fill = PINK
                cell.alignment = Alignment(horizontal="center", vertical="center")
        for ci in range(1, ws.max_column + 1):
            w = max(len(str(ws.cell(row=r, column=ci).value or "")) for r in range(1, ws.max_row+1))
            ws.column_dimensions[get_column_letter(ci)].width = min(max(w + 2, 9), 26)
        ws.freeze_panes = "D2"
        wb.save(FA_OUTPUT)
        bad = sum(1 for r in rows for c in THRESHOLDS if _is_bad(c, r.get(c)))
        print(f"\n  ✅  Saved → {FA_OUTPUT}")
        print(f"  📊  {len(rows)} stocks · {bad} cells highlighted pink")


✅ Using 57 stocks from SGX combined scan (Cell 3b).

  Fetching FA data for 57 stocks (SGX) ...

  [ 1/57] BFT      ROE=40.6%  Margin=6.2%  D/E=1.34
  [ 2/57] HQU      ROE=34.7%  Margin=15.1%  D/E=None
  [ 3/57] S68      ROE=30.1%  Margin=45.7%  D/E=0.3
  [ 4/57] OV8      ROE=26.4%  Margin=9.5%  D/E=0.3
  [ 5/57] QES      ROE=9.4%  Margin=12.4%  D/E=None
  [ 6/57] P52      ROE=18.1%  Margin=5.6%  D/E=0.16
  [ 7/57] BEC      ROE=20.3%  Margin=5.9%  D/E=0.29
  [ 8/57] ITS      ROE=68.5%  Margin=26.6%  D/E=0.1
  [ 9/57] G92      ROE=10.7%  Margin=0.7%  D/E=0.0
  [10/57] BS6      ROE=29.4%  Margin=30.3%  D/E=0.18
  [11/57] U14      ROE=3.7%  Margin=14.9%  D/E=0.28
  [12/57] 1J4      ROE=4.1%  Margin=6.2%  D/E=0.19
  [13/57] 558      ROE=9.7%  Margin=16.6%  D/E=0.02
  [14/57] I07      ROE=5.4%  Margin=1.5%  D/E=0.36
  [15/57] E28      ROE=8.6%  Margin=4.5%  D/E=0.19
  [16/57] ZBA      ROE=20.2%  Margin=14.6%  D/E=0.01
  [17/57] VC2      ROE=0.8%  Margin=1.5%  D/E=2.16
  [18/57] AWX      ROE

---
# ⏸ SGX Section Complete
**Run All stops here.** To run the S&P 500 section, select **Cell 4** below and use **Run Selected and After**.

---

In [ ]:
# ── SGX / S&P 500 boundary ───────────────────────────────────────────────
# This cell intentionally stops 'Run All' here.
# To continue with S&P 500 cells: select Cell 4 → Runtime > Run selected and after.
# ─────────────────────────────────────────────────────────────────────────
print("\n" + "═" * 60)
print("  ✅  SGX section done.")
print("  ▶   Select Cell 4 and choose  Runtime → Run selected and after")
print("      to continue with the S&P 500 section.")
print("═" * 60 + "\n")
raise KeyboardInterrupt


---
## Cell 4 — S&P 500 Download
Downloads all S&P 500 stocks + SPY benchmark. Saves to `/content/`. Run once, then use the scanners without re-downloading.

In [ ]:
# ============================================================
# S&P 500 DOWNLOAD — saves sp500_historical.csv + spy_close.csv
# Run once per session. Scanner cells load from these files.
# ============================================================
import csv, io, time, warnings
import requests, pandas as pd, yfinance as yf
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")

SP500_HIST_CSV = "/content/sp500_historical.csv"
SPY_CSV        = "/content/spy_close.csv"
BATCH_SIZE     = 50

print("=" * 55)
print("  S&P 500 DATA DOWNLOAD")
print("=" * 55)

# ── Fetch ticker list from Wikipedia ────────────────────────
print("\n[1/3] Fetching S&P 500 tickers from Wikipedia...")
html  = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
                     headers={"User-Agent": "Mozilla/5.0"}).text
sp500 = pd.read_html(io.StringIO(html), flavor="lxml")[0]
sp500["yf_ticker"] = sp500["Symbol"].str.replace(".", "-", regex=False)
sp500_ticker_info  = {
    row["yf_ticker"]: {"symbol": row["Symbol"], "name": row["Security"], "sector": row["GICS Sector"]}
    for _, row in sp500.iterrows()
}
print(f"    Found {len(sp500_ticker_info)} stocks")

# ── Download price history ───────────────────────────────────
end_date   = datetime.today().strftime("%Y-%m-%d")
start_date = (datetime.today() - timedelta(days=365)).strftime("%Y-%m-%d")
print(f"\n[2/3] Downloading data ({start_date} → {end_date})...")

print("    Fetching SPY benchmark...")
spy_raw   = yf.download("SPY", start=start_date, end=end_date, auto_adjust=True, progress=False)
spy_close = spy_raw["Close"].squeeze()
spy_close.to_csv(SPY_CSV)

all_tickers    = list(sp500_ticker_info.keys())
batches        = [all_tickers[i:i+BATCH_SIZE] for i in range(0, len(all_tickers), BATCH_SIZE)]
sp500_stock_data = {}

with open(SP500_HIST_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["symbol", "name", "sector", "date", "open", "high", "low", "close", "volume"])
    for i, batch in enumerate(batches):
        print(f"    Batch {i+1}/{len(batches)}...", end=" ", flush=True)
        try:
            raw = yf.download(batch, start=start_date, end=end_date,
                              auto_adjust=True, progress=False, threads=True)
            results = {}
            if len(batch) == 1:
                if not raw.empty: results[batch[0]] = raw.rename(columns=str.lower)
            else:
                for t in batch:
                    try:
                        df = raw.xs(t, level=1, axis=1).dropna(how="all")
                        if not df.empty: results[t] = df.rename(columns=str.lower)
                    except KeyError: pass
            for yf_t, df in results.items():
                info = sp500_ticker_info[yf_t]
                sp500_stock_data[yf_t] = df
                for date, row in df.iterrows():
                    writer.writerow([
                        info["symbol"], info["name"], info["sector"],
                        date.strftime("%Y-%m-%d"),
                        round(float(row.get("open",  0) or 0), 4),
                        round(float(row.get("high",  0) or 0), 4),
                        round(float(row.get("low",   0) or 0), 4),
                        round(float(row.get("close", 0) or 0), 4),
                        int(row.get("volume", 0) or 0),
                    ])
            print(f"ok ({len(results)}/{len(batch)})")
        except Exception as e:
            print(f"ERROR: {e}")
        time.sleep(0.3)

print(f"\n[3/3] Done!")
print(f"    Stocks downloaded : {len(sp500_stock_data)}")
print(f"    Saved to          : {SP500_HIST_CSV}")
print(f"    SPY benchmark     : {SPY_CSV}")
print("    ✅ Scanner cells can now use this data without re-downloading.")

## Cell 5 — S&P 500 Multi Signal Scanner
Scans the S&P 500 universe using **15 buy signals** from three groups:

| Group | Signals | Description |
|-------|---------|-------------|
| **EX** | EX-1, EX-2, EX-3, EX-4 | Extreme reversal bottoms (price far below SMAs, momentum accelerating down — catching capitulation bounces) |
| **MS** | MS-1,2,3,4,5,7,9,10 | Master screener momentum / trend signals (MACD, SMA alignment, volume, pullbacks) |
| **AM** | AM-11, AM-12, AM-13 | Annual macro breakouts (30-day 2nd high, 180-day high, 250-day yearly high) |

Export includes a `signals` column listing every signal that fired (e.g. `MS-3, AM-12`) and a `sig_count` column.

**Loads from memory -> CSV -> downloads fresh** (no re-download if Cell 4 was run).

In [ ]:
# ============================================================
# CELL 5 — S&P 500 MULTI SIGNAL SCANNER
# EX (1-4) + MS (1,2,3,4,5,7,9,10) + AM (11,12,13) Buy Signals
#
# Filter : volume > 500,000  AND  price > $1.00
# Loads  : memory -> sp500_historical.csv -> fresh download
# Outputs: /content/sp500_buy_signals.csv
#          /content/sp500_buy_signals_watchlist.txt
# ============================================================
import csv as _csv, io, os, time, warnings
import numpy as np, pandas as pd, requests, yfinance as yf
from datetime import datetime, timedelta
warnings.filterwarnings("ignore")

SP500_HIST_CSV = "/content/sp500_historical.csv"
SPY_CSV        = "/content/spy_close.csv"
OUTPUT_CSV     = "/content/sp500_buy_signals.csv"
OUTPUT_WL      = "/content/sp500_buy_signals_watchlist.txt"
BATCH_SIZE     = 50
MIN_VOLUME, MIN_PRICE = 500_000, 1.00

# Parameters
RSI_LEN             = 14
RSI_OS_LEVEL        = 40
LOW_TOL_PCT         = 0.5
ROC50_TOLERANCE     = -0.2
AM11_BREAKOUT_PREM  = 2.0

# ============================================================
# HELPERS
# ============================================================
def _ema(s, span):
    return s.ewm(span=span, adjust=False, min_periods=span).mean()

def _sma(s, length):
    return s.rolling(length, min_periods=length).mean()

def _rsi(s, length=14):
    d = s.diff()
    g = d.clip(lower=0); l = -d.clip(upper=0)
    ag = g.ewm(com=length-1, min_periods=length).mean()
    al = l.ewm(com=length-1, min_periods=length).mean()
    return 100 - 100 / (1 + ag / al.replace(0, np.nan))

def _macd(s, fast=12, slow=26, sig=9):
    line = _ema(s, fast) - _ema(s, slow)
    return line, _ema(line, sig), line - _ema(line, sig)

def _atr(high, low, close, length=14):
    tr = pd.concat([high - low,
                    (high - close.shift(1)).abs(),
                    (low  - close.shift(1)).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1/length, adjust=False, min_periods=length).mean()

# ============================================================
# SIGNAL DETECTOR
# ============================================================
def check_all_buy_signals(df):
    n = len(df)
    if n < 260:
        return {}

    close  = df["close"]
    open_  = df["open"]
    high   = df["high"]
    low    = df["low"]
    volume = df["volume"]

    # ── Indicators ───────────────────────────────────────────
    sma5   = _sma(close, 5)
    sma10  = _sma(close, 10)
    sma20  = _sma(close, 20)
    sma50  = _sma(close, 50)
    sma200 = _sma(close, 200)
    ema50  = _ema(close, 50)

    vol_sma = _sma(volume, 20)
    rel_vol = volume / vol_sma

    ml, sl, hl   = _macd(close, 12, 26, 9)
    _, _, vol_hl = _macd(volume, 12, 26, 9)

    rsi_val = _rsi(close, RSI_LEN)
    rsi_sma = _sma(rsi_val, 14)

    atr14    = _atr(high, low, close, 14)
    safe_atr = atr14.replace(0, 1)

    sma50_angle  = np.arctan((sma50 - sma50.shift(1)) / safe_atr) * (180 / np.pi)
    sma20_angle  = np.arctan((sma20 - sma20.shift(1)) / safe_atr) * (180 / np.pi)
    candle_angle = np.arctan((close - close.shift(1)) / safe_atr) * (180 / np.pi)

    max_oc   = pd.concat([open_, close], axis=1).max(axis=1)
    min_oc   = pd.concat([open_, close], axis=1).min(axis=1)
    uw       = high - max_oc
    lw       = min_oc - low
    body     = (close - open_).abs()

    is_green = (close > open_)
    is_red   = (close < open_)

    roc10  = (sma10 - sma10.shift(1)) / sma10.shift(1) * 100
    roc20  = (sma20 - sma20.shift(1)) / sma20.shift(1) * 100
    roc50  = (sma50 - sma50.shift(1)) / sma50.shift(1) * 100
    roc_c1 = (close - close.shift(1)) / close.shift(1) * 100
    roc_s10= roc10
    roc_l1 = (low   - low.shift(1))   / low.shift(1)   * 100

    mom_acc_up   = (sma10 - sma20) >= (sma20 - sma50)
    mom_acc_down = (sma20 - sma10) >= (sma50 - sma20)

    is_5d_low = (low == low.rolling(5).min())
    pull5     = (is_5d_low.shift(1).fillna(False) |
                 is_5d_low.shift(2).fillna(False))

    # ── Scalar extractor helpers ─────────────────────────────
    def f(s, k=0):
        val = s.iloc[-1-k]
        return 0.0 if pd.isna(val) else float(val)

    def flag(s, k=0):
        val = s.iloc[-1-k]
        return False if pd.isna(val) else bool(val)

    # Raw arrays
    C = close.values; O = open_.values; H = high.values; L = low.values; V = volume.values

    # Current bar
    c0=C[-1]; o0=O[-1]; h0=H[-1]; l0=L[-1]; v0=V[-1]
    uw0=f(uw); lw0=f(lw); bd0=f(body)
    s10_0=f(sma10); s20_0=f(sma20); s50_0=f(sma50); s200_0=f(sma200)
    s10_1=f(sma10,1); s20_1=f(sma20,1); s50_1=f(sma50,1)
    s20_2=f(sma20,2); s20_3=f(sma20,3); s20_4=f(sma20,4)
    e50_0=f(ema50); e50_1=f(ema50,1); e50_5=f(ema50,5)
    s50_5=f(sma50,5)
    ml0=f(ml); sl0=f(sl); hl0=f(hl); hl1=f(hl,1)
    vhl0=f(vol_hl); vhl1=f(vol_hl,1)
    rs0=f(rsi_val); rsm0=f(rsi_sma); rsm1=f(rsi_sma,1)
    rsm2=f(rsi_sma,2); rsm3=f(rsi_sma,3)
    rv0=f(rel_vol); vs0=f(vol_sma); vs1=f(vol_sma,1)
    dc0 = (c0 - C[-2]) / C[-2] * 100 if C[-2] != 0 else 0
    r10_0=f(roc10); r20_0=f(roc20); r50_0=f(roc50)
    r10_1=f(roc10,1); r20_1=f(roc20,1); r50_1=f(roc50,1)
    rc1_0=f(roc_c1); rs10_0=f(roc_s10); rl1_0=f(roc_l1)
    ang50=f(sma50_angle); ang20=f(sma20_angle); angc=f(candle_angle)
    green0=c0>o0; red0=c0<o0; ng0=c0<=o0
    new_hi0=h0>H[-2]; new_lo0=l0<L[-2]
    ex1_d0=s10_0-l0; ex1_d1=s10_1-L[-2]
    pull5_0=flag(pull5); roc50ok=(r50_0>=ROC50_TOLERANCE)
    mad0=flag(mom_acc_down); mau0=flag(mom_acc_up)

    # ── EX BUY ───────────────────────────────────────────────
    # EX-1
    buyEX1 = (
        (c0 < s10_0 < s20_0 < s50_0) and
        new_lo0 and (ex1_d0 > ex1_d1) and (o0 < C[-2]) and
        red0 and (lw0 > bd0*0.6) and (l0 < s10_0*0.85) and
        mad0
    )

    # EX-2
    buyEX2 = (
        (c0 < s10_0) and (s10_0 < s20_0) and
        new_lo0 and (l0 < s10_0*0.85) and (lw0 > bd0*0.6) and
        mad0
    )

    # EX-3
    ex3_vol   = (v0 > vs0*1.02) and (V[-2] > vs1)
    ex3_price = new_lo0 and (abs(rc1_0) < 2)
    ex3_ext   = (c0 < s10_0*0.97) and (rc1_0 < rs10_0)
    ex3_ma    = (abs(s10_0 - s20_0) / s20_0) <= 0.05 if s20_0 != 0 else False
    ex3_roc   = rl1_0 <= rs10_0
    ex3_green = any(C[-1-k] > O[-1-k] and C[-1-k] > C[-2-k] for k in range(4))
    buyEX3 = (
        (c0 < s10_0 < s20_0) and
        ex3_vol and ex3_price and ex3_ext and ex3_ma and ex3_roc and ex3_green and
        mad0
    )

    # EX-4
    buyEX4 = (
        (ml0 < 0) and (ml0 > sl0) and
        (C[-2] < s20_1) and (c0 > s20_0) and
        (c0 < s20_0*1.02) and (s20_0 >= s20_1) and
        mad0
    )

    # ── MS BUY ───────────────────────────────────────────────
    # MS-1
    buyMS1 = (
        (c0 < s10_0 < s20_0) and
        ((o0 - O[-2]) - (c0 - C[-2]) < 0) and
        (hl0 > hl1) and (hl0 >= 0) and
        ((rv0 > 1.5) or (green0 and uw0 < bd0*0.5)) and
        (rs0 < RSI_OS_LEVEL) and (abs(rs0 - rsm0) <= 9) and
        (l0 <= L[-2] * (1 + LOW_TOL_PCT/100))
    )

    # MS-2 (first occurrence)
    ms2_raw = (
        (is_green) &
        (sma10 > sma10.shift(1)) & (sma20 > sma20.shift(1)) & (roc50 >= ROC50_TOLERANCE) &
        (roc10 > roc20) & (roc20 > roc50) &
        (ema50 > ema50.shift(5)) &
        (high == high.rolling(5).max()) &
        pull5
    )
    buyMS2 = flag(ms2_raw) and not flag(ms2_raw, 1)

    # MS-3
    buyMS3 = (
        (s20_0 > s50_0) and (s50_0 >= s50_1) and
        (C[-2] <= s20_1*1.01) and (C[-2] >= s50_1) and
        (c0 > s20_0) and green0 and (c0 > O[-2]) and (uw0 <= bd0) and (c0 > s10_0*0.9) and
        (rv0 > 1.1) and (vhl0 >= vhl1) and
        (rsm0 >= rsm3) and
        not (C[-2]<s20_1 and C[-3]<s20_2 and C[-4]<s20_3 and C[-5]<s20_4) and
        ((dc0 <= 5) or (c0 > s10_0))
    )

    # MS-4
    ms4_gap = abs(s20_0 - s50_0) / s50_0 * 100 if s50_0 != 0 else 99
    buyMS4 = (
        (0 <= ang20 <= 15) and (0 <= ang50 <= 10) and
        (((s20_0 < s50_0) and (ms4_gap <= 1.0) and (o0 < s50_0)) or
         ((s20_0 >= s50_0) and (ms4_gap <= 2.0) and (c0 > s20_0) and (o0 < s20_0*1.01))) and
        (rv0 > 1.2) and (v0 > V[-2]) and
        green0 and
        (rsm0 >= rsm2 * 1.01) and
        (h0 == float(high.rolling(5).max().iloc[-1])) and
        (s50_0 >= s50_5 * 0.99)
    )

    # MS-5
    h40_prev = float(high.shift(1).rolling(40).max().iloc[-1])
    buyMS5 = (
        (c0 > h40_prev) and
        (uw0 <= bd0*0.3) and green0 and
        (rv0 > 2.0) and
        (s10_0 > s20_0 > s50_0) and
        (hl0 > 0) and (hl0 > hl1) and
        (angc <= 50)
    )

    # MS-7
    ms7_avg = (s10_0 + s20_0 + s50_0) / 3
    ms7_reds = sum(1 for k in range(1, 8) if C[-1-k] < O[-1-k])
    h7_prev = float(high.shift(1).rolling(7).max().iloc[-1])
    s20_5 = f(sma20, 5)
    buyMS7 = (
        (s10_0 > s20_0 > s50_0) and (s50_0 >= s50_1) and
        ms7_avg != 0 and
        (abs(s10_0 - ms7_avg)/ms7_avg <= 0.02) and
        (abs(s20_0 - ms7_avg)/ms7_avg <= 0.02) and
        (abs(s50_0 - ms7_avg)/ms7_avg <= 0.02) and
        green0 and (l0 > s20_0) and
        (ms7_reds >= 3) and
        (c0 > h7_prev) and
        (s20_0 >= s20_5 * 1.03)
    )

    # MS-9 (first occurrence)
    ms9_g2050  = sma20 - sma50
    ms9_g50200 = sma50 - sma200
    ms9_today = (ms9_g2050 > ms9_g2050.shift(1)) & (ms9_g50200 > ms9_g50200.shift(1))
    ms9_yest  = (ms9_g2050.shift(1) > ms9_g2050.shift(2)) & (ms9_g50200.shift(1) > ms9_g50200.shift(2))
    buyMS9 = flag(ms9_today) and not flag(ms9_yest) and pull5_0

    # MS-10 (first occurrence)
    acc10  = roc10 - roc10.shift(1)
    acc20  = roc20 - roc20.shift(1)
    acc50  = roc50 - roc50.shift(1)
    ms10_acc = (acc10 > acc20) & (acc20 > acc50)
    ms10_up  = (acc10 > 0) & (roc10 > 0)
    buyMS10 = flag(ms10_acc) and not flag(ms10_acc, 1) and flag(ms10_up) and roc50ok and pull5_0

    # ── AM BUY ───────────────────────────────────────────────
    # AM-11: 30-Day 2nd High Breakout
    buyAM11 = False
    if n >= 35:
        h_30 = H[-31:-1]
        h1_val = h_30.max()
        s_high = 0.0; s_idx = 1
        for i in range(1, 31):
            if H[-1-i] > s_high and H[-1-i] < h1_val:
                s_high = H[-1-i]; s_idx = i
        if s_high > 0 and 1 < s_idx < 30:
            am11_peak = (H[-1-s_idx] > H[-2-s_idx]) and (H[-1-s_idx] > H[-s_idx])
            look = max(1, s_idx - 1)
            low_aft = L[-look:].min()
            am11_base = green0 and (c0 > s_high) and (c0 <= s_high * (1 + AM11_BREAKOUT_PREM/100))
            buyAM11 = am11_base and am11_peak and (low_aft < L[-1-s_idx])

    # AM-12: 180-Day High Breakout
    buyAM12 = False
    if n >= 185:
        h_180 = H[-181:-1]
        am12_prev = h_180.max()
        am12_off  = 180 - int(np.argmax(h_180))
        buyAM12 = (
            green0 and (h0 > am12_prev) and
            (am12_prev >= h0 * 0.95) and (am12_off >= 10) and
            (uw0 <= bd0 * 0.10) and
            (c0 > s20_0 > s50_0)
        )

    # AM-13: 250-Day Yearly High Breakout
    buyAM13 = False
    if n >= 255:
        h_250 = H[-251:-1]
        am13_prev = h_250.max()
        am13_off  = 250 - int(np.argmax(h_250))
        buyAM13 = (
            green0 and (h0 > am13_prev) and
            (am13_prev >= h0 * 0.95) and (am13_off >= 10) and
            (uw0 <= bd0 * 0.10) and
            (c0 > s20_0 > s50_0)
        )

    return {
        "EX-1":  bool(buyEX1),  "EX-2":  bool(buyEX2),
        "EX-3":  bool(buyEX3),  "EX-4":  bool(buyEX4),
        "MS-1":  bool(buyMS1),  "MS-2":  bool(buyMS2),
        "MS-3":  bool(buyMS3),  "MS-4":  bool(buyMS4),
        "MS-5":  bool(buyMS5),  "MS-7":  bool(buyMS7),
        "MS-9":  bool(buyMS9),  "MS-10": bool(buyMS10),
        "AM-11": bool(buyAM11), "AM-12": bool(buyAM12),
        "AM-13": bool(buyAM13),
    }


# ============================================================
# DATA LOADING  (memory -> CSV -> fresh download)
# ============================================================
print("=" * 65)
print("  S&P 500 MULTI SIGNAL SCANNER  (EX + MS + AM)")
print("=" * 65)

try:
    _ = sp500_stock_data, sp500_ticker_info
    print("\n  Using in-memory S&P 500 data.")
except NameError:
    if os.path.exists(SP500_HIST_CSV):
        print("\n  Loading from CSV...")
        df_all = pd.read_csv(SP500_HIST_CSV, parse_dates=["date"])
        sp500_ticker_info = {}; sp500_stock_data = {}
        for sym, grp in df_all.groupby("symbol"):
            yf_t = sym.replace(".", "-")
            grp  = grp.set_index("date").sort_index()
            sp500_stock_data[yf_t]  = grp[["open", "high", "low", "close", "volume"]]
            sp500_ticker_info[yf_t] = {"symbol": sym, "name": grp["name"].iloc[0], "sector": grp["sector"].iloc[0]}
        print(f"  Loaded {len(sp500_stock_data)} stocks.")
    else:
        print("\n  Downloading S&P 500 data (run Cell 4 first)...")
        end_date   = datetime.today().strftime("%Y-%m-%d")
        start_date = (datetime.today() - timedelta(days=365)).strftime("%Y-%m-%d")
        html  = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
                             headers={"User-Agent": "Mozilla/5.0"}).text
        sp500 = pd.read_html(io.StringIO(html), flavor="lxml")[0]
        sp500["yf_ticker"] = sp500["Symbol"].str.replace(".", "-", regex=False)
        sp500_ticker_info  = {
            row["yf_ticker"]: {"symbol": row["Symbol"], "name": row["Security"], "sector": row["GICS Sector"]}
            for _, row in sp500.iterrows()
        }
        spy_raw   = yf.download("SPY", start=start_date, end=end_date, auto_adjust=True, progress=False)
        spy_close = spy_raw["Close"].squeeze()
        spy_close.to_csv(SPY_CSV)
        all_tickers = list(sp500_ticker_info.keys())
        batches = [all_tickers[i:i+BATCH_SIZE] for i in range(0, len(all_tickers), BATCH_SIZE)]
        sp500_stock_data = {}
        for i, batch in enumerate(batches):
            print(f"  Batch {i+1}/{len(batches)}...", end=" ", flush=True)
            try:
                raw = yf.download(batch, start=start_date, end=end_date,
                                  auto_adjust=True, progress=False, threads=True)
                if len(batch) == 1:
                    if not raw.empty: sp500_stock_data[batch[0]] = raw.rename(columns=str.lower)
                else:
                    for t in batch:
                        try:
                            df = raw.xs(t, level=1, axis=1).dropna(how="all")
                            if not df.empty: sp500_stock_data[t] = df.rename(columns=str.lower)
                        except KeyError: pass
                print(f"ok ({sum(1 for t in batch if t in sp500_stock_data)}/{len(batch)})")
            except Exception as e: print(f"ERROR: {e}")
            time.sleep(0.3)

# ============================================================
# SCAN
# ============================================================
print(f"\n  Scanning {len(sp500_stock_data)} stocks...")

confirmed = []

for yf_t, df in sp500_stock_data.items():
    info       = sp500_ticker_info.get(yf_t, {})
    last_close = float(df["close"].iloc[-1])
    last_vol   = float(df["volume"].iloc[-1])

    if last_vol < MIN_VOLUME or last_close < MIN_PRICE:
        continue

    try:
        sigs = check_all_buy_signals(df)
    except Exception:
        continue

    fired = [name for name, val in sigs.items() if val]
    if not fired:
        continue

    confirmed.append({
        "symbol":     info.get("symbol", yf_t),
        "name":       info.get("name", ""),
        "sector":     info.get("sector", ""),
        "signals":    ", ".join(fired),
        "sig_count":  len(fired),
        "last_close": round(last_close, 4),
        "last_vol":   int(last_vol),
        "last_date":  df.index[-1].strftime("%Y-%m-%d"),
    })

confirmed.sort(key=lambda r: (-r["sig_count"], r["symbol"]))

# ============================================================
# OUTPUT
# ============================================================
print("\n" + "=" * 95)
print(f"  S&P 500 MULTI SIGNAL RESULTS -- {len(confirmed)} stocks matched")
print("  EX=Extreme Reversal  MS=Master Screener  AM=Annual Macro Breakout")
print("=" * 95)
if confirmed:
    print(f"  {'Symbol':<8} {'Name':<28} {'Sector':<22} {'Close':>8} {'Signals'}")
    print(f"  {'-'*8} {'-'*28} {'-'*22} {'-'*8} {'-'*30}")
    for r in confirmed:
        print(f"  {r['symbol']:<8} {r['name'][:28]:<28} {r['sector'][:22]:<22} "
              f"{r['last_close']:>8.2f}  {r['signals']}")
else:
    print("  No stocks matched today.")
print("=" * 95)

fields = ["symbol", "name", "sector", "signals", "sig_count", "last_close", "last_vol", "last_date"]
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = _csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(confirmed)

with open(OUTPUT_WL, "w", encoding="utf-8") as f:
    for r in confirmed:
        f.write(f"{r['symbol']}\n")

print(f"\n  Results -> {OUTPUT_CSV}  ({len(confirmed)} rows)")
print(f"  TV list -> {OUTPUT_WL}  ({len(confirmed)} tickers)")

sp500_matched = confirmed


## Cell 5b-FA — S&P 500 FA Check
Fundamental analysis for stocks from the combined crossup+RSI scan.
Fetches 10 FA indicators via yfinance · exports to Excel with pink highlighting for bad metrics.

In [ ]:
# ============================================================
# S&P 500 FA CHECK  (Cell 5b-FA)
# Reads confirmed stocks from Cell 5 (S&P 500 Scanner)
# Output: /content/sp500_fa_check.xlsx
# ============================================================
import pandas as pd, yfinance as yf, time, os, warnings
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
warnings.filterwarnings("ignore")

FA_OUTPUT = "/content/sp500_fa_check.xlsx"

THRESHOLDS = {
    "EPS Growth (%)": (">=", 10),
    "Rev Growth (%)": (">",  10),
    "ROE (%)":        (">",  15),
    "ROIC (%)":       (">=", 15),
    "P/FCF":          ("<",  25),
    "TTM PE":         None,
    "Forward PE":     None,
    "Net Margin (%)": (">",  9.5),
    "Debt/Equity":    ("<",  1.0),
    "Rule of 40 (%)": (">",  40),
}
PINK = PatternFill(start_color="FFB6C1", end_color="FFB6C1", fill_type="solid")

def _is_bad(col, val):
    if val is None or THRESHOLDS.get(col) is None:
        return False
    op, thr = THRESHOLDS[col]
    try:
        v = float(val)
        return v < thr if op in (">=", ">") else v >= thr
    except: return False

def fetch_fa(yf_ticker):
    res = {c: None for c in THRESHOLDS}
    try:
        t    = yf.Ticker(yf_ticker)
        info = t.info
        eg   = info.get("earningsGrowth");  rg = info.get("revenueGrowth")
        roe  = info.get("returnOnEquity");  mg = info.get("profitMargins")
        de   = info.get("debtToEquity")
        ttm  = info.get("trailingPE");      fwd = info.get("forwardPE")
        mc   = info.get("marketCap");       fcf = info.get("freeCashflow")
        res["EPS Growth (%)"] = round(eg * 100, 1)  if eg  is not None else None
        res["Rev Growth (%)"] = round(rg * 100, 1)  if rg  is not None else None
        res["ROE (%)"]        = round(roe* 100, 1)  if roe is not None else None
        res["Net Margin (%)"] = round(mg * 100, 1)  if mg  is not None else None
        res["Debt/Equity"]    = round(de / 100, 2)  if de  is not None else None
        res["TTM PE"]         = round(ttm, 1)        if ttm is not None else None
        res["Forward PE"]     = round(fwd, 1)        if fwd is not None else None
        res["P/FCF"]          = round(mc / fcf, 1)  if (mc and fcf and fcf > 0) else None
        try:
            bs = t.balance_sheet; fi = t.financials
            if not bs.empty and not fi.empty:
                ni_r  = [r for r in fi.index if "Net Income" in str(r)]
                eq_r  = [r for r in bs.index if any(k in str(r) for k in ("Stockholder","Common Stock Equity","Total Equity"))]
                ltd_r = [r for r in bs.index if "Long Term Debt" in str(r)]
                if ni_r and eq_r:
                    ni  = float(fi.loc[ni_r[0]].iloc[0])
                    eq  = float(bs.loc[eq_r[0]].iloc[0])
                    ltd = float(bs.loc[ltd_r[0]].iloc[0]) if ltd_r else 0
                    ic  = eq + ltd
                    if ic > 0: res["ROIC (%)"] = round(ni / ic * 100, 1)
        except: pass
        if res["Rev Growth (%)"] is not None and res["Net Margin (%)"] is not None:
            res["Rule of 40 (%)"] = round(res["Rev Growth (%)"] + res["Net Margin (%)"], 1)
    except: pass
    return res

# ── Load scan list ────────────────────────────────────────────
try:
    scan_list = sp500_matched
    print(f"✅ Using {len(scan_list)} stocks from S&P 500 Scanner (Cell 5).")
except NameError:
    csv_path = "/content/sp500_scan_results.csv"
    if os.path.exists(csv_path):
        _df = pd.read_csv(csv_path)
        scan_list = _df[_df["signal_confirmed"].astype(str).str.lower() == "true"].to_dict("records")
        print(f"📂 Loaded {len(scan_list)} confirmed stocks from {csv_path}.")
    else:
        print("❌ No data found — run Cell 5 first."); scan_list = []

# ── FA Fetch ──────────────────────────────────────────────────
if not scan_list:
    print("Nothing to analyse.")
else:
    print(f"\n  Fetching FA data for {len(scan_list)} stocks (S&P 500) ...\n")
    rows = []
    for i, r in enumerate(scan_list):
        sym = r["symbol"]; yf_t = sym.replace(".", "-")
        print(f"  [{i+1:>3}/{len(scan_list)}] {sym:<8}", end=" ", flush=True)
        fa  = fetch_fa(yf_t)
        row = {"Symbol": sym, "Name": r.get("name","")[:24], "Sector": r.get("sector","")[:20],
               "Close": r.get("last_close"), **fa}
        rows.append(row); time.sleep(0.4)
        print(f"ROE={fa.get('ROE (%)')}%  Margin={fa.get('Net Margin (%)')}%  D/E={fa.get('Debt/Equity')}")

    def _passes_filter(row):
        de    = row.get("Debt/Equity")
        nm    = row.get("Net Margin (%)")
        roic  = row.get("ROIC (%)")
        rev_g = row.get("Rev Growth (%)")
        eps_g = row.get("EPS Growth (%)")
        if any(v is None for v in [de, nm, roic, rev_g, eps_g]):
            return False
        return de < 1 and nm >= 10 and roic > 5 and rev_g > 0 and eps_g > 0

    before = len(rows)
    rows   = [r for r in rows if _passes_filter(r)]
    print(f"\n  FA Filter: {before} → {len(rows)} stocks pass (D/E<1, Margin≥10%, ROIC>5%, RevGrow>0%, EPS>0%)")

    if not rows:
        print("  No stocks passed the FA filter.")
    else:
        df = pd.DataFrame(rows)
        df.to_excel(FA_OUTPUT, index=False, engine="openpyxl")
        wb = load_workbook(FA_OUTPUT); ws = wb.active; ws.title = "SP500 FA"
        for c in range(1, ws.max_column + 1):
            cell = ws.cell(row=1, column=c)
            cell.fill = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
            cell.font = Font(bold=True, color="FFFFFF", size=10)
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        ws.row_dimensions[1].height = 32
        col_names = list(df.columns)
        for ri in range(2, ws.max_row + 1):
            for ci, cname in enumerate(col_names, 1):
                cell = ws.cell(row=ri, column=ci)
                if _is_bad(cname, cell.value): cell.fill = PINK
                cell.alignment = Alignment(horizontal="center", vertical="center")
        for ci in range(1, ws.max_column + 1):
            w = max(len(str(ws.cell(row=r, column=ci).value or "")) for r in range(1, ws.max_row+1))
            ws.column_dimensions[get_column_letter(ci)].width = min(max(w + 2, 9), 26)
        ws.freeze_panes = "E2"
        wb.save(FA_OUTPUT)
        bad = sum(1 for r in rows for c in THRESHOLDS if _is_bad(c, r.get(c)))
        print(f"\n  ✅  Saved → {FA_OUTPUT}")
        print(f"  📊  {len(rows)} stocks · {bad} cells highlighted pink")


## Cell 6 — Combine Watchlists + Download All Results
Merges every scanner's output into two combined CSVs (one for SGX, one for S&P 500),
then downloads all individual and combined files to your local drive.

**SGX combined** → `sgx_combined_watchlist.csv`  
**S&P 500 combined** → `spy_combined_watchlist.csv`

Each combined CSV lists every unique ticker, which scanners found it, and (for Master Buy stocks) which signals fired.

In [ ]:
# ============================================================
# CELL 6 — COMBINE WATCHLISTS + DOWNLOAD ALL RESULTS
#
# Merges all SGX scanner CSVs into one combined watchlist.
# Merges all S&P 500 scanner CSVs into one combined watchlist.
# Then downloads every file to your local drive.
#
# Combined SGX  → /content/sgx_combined_watchlist.csv
# Combined SPY  → /content/spy_combined_watchlist.csv
# ============================================================
import os
import pandas as pd
from google.colab import files

SGX_COMBINED_CSV = "/content/sgx_combined_watchlist.csv"
SPY_COMBINED_CSV = "/content/spy_combined_watchlist.csv"

# Source CSVs: (file_path, symbol_column, scanner_label)
SGX_SOURCES = [
    ("/content/sgx_buy_signals.csv",           "symbol", "Master Buy (A/C/D)"),
]
SPY_SOURCES = [
    ("/content/sp500_scan_results.csv",        "symbol", "SMA Gap"),
    ("/content/sp500_buy_signals.csv",         "symbol", "Multi Signal (EX/MS/AM)"),
]

ALL_DOWNLOAD_FILES = [
    # ── SGX individual ────────────────────────────────────────
    "/content/sgx_fa_check.xlsx",
    "/content/sgx_buy_signals.csv",
    "/content/sgx_buy_signals_watchlist.txt",
    SGX_COMBINED_CSV,
    # ── S&P 500 individual ────────────────────────────────────
    "/content/sp500_fa_check.xlsx",
    "/content/sp500_scan_results.csv",
    "/content/sp500_watchlist.txt",
    "/content/sp500_buy_signals.csv",
    "/content/sp500_buy_signals_watchlist.txt",
    SPY_COMBINED_CSV,
]

# ════════════════════════════════════════════════════════════
# BUILD COMBINED WATCHLISTS
# ════════════════════════════════════════════════════════════
def build_combined(sources, out_path, market_label):
    merged = {}  # symbol -> dict
    for path, sym_col, source_name in sources:
        if not os.path.exists(path):
            print(f"  ⚠️  {os.path.basename(path)} not found — skipping")
            continue
        try:
            df = pd.read_csv(path)
            if "signal_confirmed" in df.columns:
                df = df[df["signal_confirmed"].astype(str).str.lower() == "true"]
            if sym_col not in df.columns:
                print(f"  ⚠️  No '{sym_col}' column in {os.path.basename(path)}")
                continue
            for _, row in df.iterrows():
                sym = str(row[sym_col]).strip()
                if not sym or sym == "nan":
                    continue
                if sym not in merged:
                    merged[sym] = {
                        "symbol":      sym,
                        "name":        str(row.get("name", "")).strip(),
                        "market_sector": str(row.get("market", row.get("sector", ""))).strip(),
                        "sources":     set(),
                        "buy_signals": "",
                    }
                merged[sym]["sources"].add(source_name)
                if "signals" in df.columns and not pd.isna(row.get("signals", float("nan"))):
                    merged[sym]["buy_signals"] = str(row["signals"])
            print(f"  ✅ {source_name:<20} {len(df):>4} rows")
        except Exception as e:
            print(f"  ❌ {os.path.basename(path)}: {e}")

    rows = [
        {
            "symbol":        d["symbol"],
            "name":          d["name"],
            "market_sector": d["market_sector"],
            "sources":       ", ".join(sorted(d["sources"])),
            "buy_signals":   d["buy_signals"],
        }
        for d in sorted(merged.values(), key=lambda x: x["symbol"])
    ]
    if rows:
        pd.DataFrame(rows).to_csv(out_path, index=False)
        print(f"  → {len(rows)} unique tickers  →  {os.path.basename(out_path)}")
    else:
        print(f"  (no data yet — run the {market_label} scanner cells first)")
    return rows

print("=" * 60)
print("  BUILDING COMBINED WATCHLISTS")
print("=" * 60)

print("\n── SGX ─────────────────────────────────────────────────")
sgx_rows = build_combined(SGX_SOURCES, SGX_COMBINED_CSV, "SGX")

print("\n── S&P 500 (SPY) ────────────────────────────────────────")
spy_rows = build_combined(SPY_SOURCES, SPY_COMBINED_CSV, "S&P 500")

# ════════════════════════════════════════════════════════════
# DOWNLOAD ALL FILES
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  DOWNLOADING RESULTS TO LOCAL DRIVE")
print("=" * 60)

ok, skipped = [], []
for path in ALL_DOWNLOAD_FILES:
    if os.path.exists(path):
        print(f"  ⬇️  {os.path.basename(path)}")
        files.download(path)
        ok.append(path)
    else:
        skipped.append(path)
        print(f"  ⚠️  Not found: {os.path.basename(path)}")

print(f"\n  Downloaded : {len(ok)} file(s)")
if skipped:
    print(f"  Skipped    : {len(skipped)} — run missing scanner cells first")
